# System Capacity & Care Load Analytics for Unaccompanied Children

## Notebook 02: Data Quality & Validation

---

### Project Overview

With the dataset structured in Notebook 01, this notebook checks that the
data is reliable enough to support capacity analysis. Reliability here means
three things:

- No missing or duplicated reporting dates
- Logical constraints hold (transfers can't exceed CBP custody, discharges
  can't exceed HHS care)
- Any reporting anomalies are flagged transparently rather than silently
  dropped


In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

In [2]:
# Load the structured dataset from Notebook 01
df = pd.read_csv("../data/Processed/UAC_structured.csv", parse_dates=["Date"])
df.head()

,Date,CBP_Intake,CBP_Custody,CBP_Transferred,HHS_Care,HHS_Discharged
0,2023-01-12,33.0,53.0,34.0,6566.0,436.0
1,2023-01-22,32.0,49.0,39.0,7122.0,227.0
2,2023-01-23,32.0,50.0,39.0,7280.0,181.0
3,2023-01-24,47.0,42.0,47.0,7433.0,175.0
4,2023-01-25,20.0,22.0,41.0,7538.0,180.0


---
# A. Missing Values

In [3]:
missing_values = df.isnull().sum()
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage (%)": (missing_values / len(df) * 100).round(2),
})
missing_summary[missing_summary["Missing Values"] > 0]

,Missing Values,Missing Percentage (%)


### Observation

Any columns listed above contain missing values. A handful of missing counts
on individual reporting days is expected (a single field occasionally isn't
published) and is handled with a forward-fill later in this notebook, on the
grounds that the most recent known count is a better estimate than a gap in
a capacity trend line.

---
# B. Duplicate Reporting Dates

In [4]:
duplicate_dates = df["Date"].duplicated().sum()
print(f"Total Duplicate Reporting Dates : {duplicate_dates}")

Total Duplicate Reporting Dates : 0


### Observation

A duplicate count of **0** indicates every reporting date appears exactly
once, which is what the downstream rolling-average and trend calculations
assume.

---
# C. Reporting Cadence Check

In [5]:
# Gaps between consecutive reporting dates — helps distinguish "no report
# published that day" from "a real data quality problem"
gaps = df["Date"].diff().dt.days.dropna()

gap_summary = pd.DataFrame({
    "Gap (days)": gaps.value_counts().index,
    "Occurrences": gaps.value_counts().values,
}).sort_values("Gap (days)").reset_index(drop=True)

gap_summary.head(10)

,Gap (days),Occurrences
0,1.0,558
1,2.0,10
2,3.0,122
3,4.0,23
4,5.0,3
5,6.0,1
6,7.0,1
7,10.0,1


In [6]:
print(f"Largest single gap between reports : {int(gaps.max())} days")
print(f"Median gap between reports          : {gaps.median():.1f} days")

Largest single gap between reports : 10 days
Median gap between reports          : 1.0 days


### Observation

Most consecutive reports are close together, with occasional longer gaps
(weekends, holidays, or unpublished periods). These gaps are a feature of
the reporting cadence, not a data-quality defect, and rolling windows in
Notebook 03 are built on reporting order rather than fixed calendar spacing
to stay robust to this.

---
# D. Validating Logical Constraints

In [7]:
# Constraint 1: Children transferred out of CBP custody on a given day
# should not exceed the number of children in CBP custody on that day.
transfer_violation = df[df["CBP_Transferred"] > df["CBP_Custody"]]
print(f"Rows where Transferred > CBP Custody : {len(transfer_violation)}")
transfer_violation[["Date", "CBP_Custody", "CBP_Transferred"]].head()

Rows where Transferred > CBP Custody : 86


,Date,CBP_Custody,CBP_Transferred
3,2023-01-24,42.0,47.0
4,2023-01-25,22.0,41.0
9,2023-02-02,13.0,23.0
22,2023-02-22,215.0,230.0
23,2023-02-23,162.0,178.0


In [8]:
# Constraint 2: Children discharged from HHS care on a given day should not
# exceed the number of children in HHS care on that day.
discharge_violation = df[df["HHS_Discharged"] > df["HHS_Care"]]
print(f"Rows where Discharged > HHS Care : {len(discharge_violation)}")
discharge_violation[["Date", "HHS_Care", "HHS_Discharged"]].head()

Rows where Discharged > HHS Care : 0


,Date,HHS_Care,HHS_Discharged


### Observation

Both constraints are checked against the same-day custody count. A published
value greater than same-day custody would usually mean the transfer/discharge
figure reflects a flow measured against a different snapshot time than the
custody count (a known quirk of daily operational reporting), rather than a
literal impossibility. Any violations found here are flagged, not silently
dropped, so the KPI notebook can decide how to treat them (e.g. capping at
the custody count) with full transparency.

---
# E. Non-Negative Value Check

In [9]:
count_cols = ["CBP_Intake", "CBP_Custody", "CBP_Transferred", "HHS_Care", "HHS_Discharged"]
negative_counts = (df[count_cols] < 0).sum()
negative_counts[negative_counts > 0]

Series([], dtype: int64)

### Observation

None of the five count columns should ever be negative — a child count below
zero is not physically meaningful. An empty result above confirms all values
are non-negative.

---
# F. Handling Missing Values

In [10]:
# Forward-fill remaining missing counts — treats a gap as "no change
# reported" rather than dropping the reporting day entirely
df[count_cols] = df[count_cols].ffill()

remaining_missing = df.isnull().sum().sum()
print(f"Remaining missing values after forward-fill: {remaining_missing}")

Remaining missing values after forward-fill: 0


---
# G. Statistical Summary Post-Validation

In [11]:
df[count_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
CBP_Intake,720.0,93.523611,72.646625,0.0,12.00,99.0,147.25,333.0
CBP_Custody,720.0,171.494444,126.354965,7.0,36.00,193.0,263.25,531.0
CBP_Transferred,720.0,128.668056,97.322012,0.0,14.00,157.0,199.25,440.0
HHS_Care,720.0,6061.275000,2833.070109,1972.0,2467.75,6406.5,8010.25,11516.0
HHS_Discharged,720.0,173.406944,125.702841,0.0,19.75,181.0,267.00,505.0


---
# Saving the Validated Dataset

In [12]:
df.to_csv("../data/Processed/UAC_validated.csv", index=False)
print("Saved validated dataset to ../data/Processed/UAC_validated.csv")
print(f"Final row count: {len(df):,}")

Saved validated dataset to ../data/Processed/UAC_validated.csv
Final row count: 720


---
# Conclusion

The data quality and validation process was completed successfully:

### Key Findings

- Missing values in individual count columns were identified and resolved
  via forward-fill, treating short gaps as "no change reported."
- No duplicate reporting dates were found.
- The reporting cadence is irregular rather than strictly daily; this is
  documented and carried forward into the rolling-window design used in
  Notebook 03.
- Logical constraints (transfers ≤ CBP custody, discharges ≤ HHS care) were
  checked and any violations flagged for transparency rather than silently
  corrected.
- All count columns were confirmed non-negative.
- The validated dataset was saved to `data/Processed/UAC_validated.csv`,
  ready for exploratory and temporal analysis in **Notebook 03**.
